# Validate the 2nd order models (fine-tuned against the first-order consensus) against real data

This notebook documents the validation of the 2nd-order models against a test set of daily rainfall images manually transcribed and provided by Ciara Ryan. It takes the model checkpoints created by ```finetune_1st_order_models.ipynb```, runs an extraction job for the test data from each model (on Azure), downloads the extracted transcriptions, and makes validation summaries comparing the transcriptions with the ground truth for the test data.

After completing this notebook, we know how well the 2nd-order models do on the test data. In particular, we can see improvements compared to the 1st-order models.

In [7]:
import json
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from posixpath import normpath

# Define model settings for validation jobs.
# Each entry: (label, model_slug, batch_size, total_shards)
MODEL_SETTINGS = [
    ("SmolVLM", "smolvlm2", 50, 1),
    ("Granite4", "granite4", 50, 1),
    ("Gemma3", "gemma3", 50, 1),
    ("Gemma4", "gemma4", 50, 1),
    ("Ministral", "ministral", 15, 1),
]

# ND96amsr A100 v4 has 8 GPUs per node. Use one extraction worker per GPU.
NODE_GPU_WORKERS = 8

# Dataset the models were trained against. 
#  Must match the dataset used in finetune_1st_order_models.ipynb
DATASET_DIR = "consensus_data/1st_order"
TRAINING_IMAGES_PATH = f"{DATASET_DIR.rstrip('/')}/images"

print(f"Checkpoint training dataset path: {TRAINING_IMAGES_PATH}")
print(f"Node GPU workers per extraction job: {NODE_GPU_WORKERS}")

Checkpoint training dataset path: consensus_data/1st_order/images
Node GPU workers per extraction job: 8


## Auto-discover 2nd-order checkpoints

Look up the latest fine-tuned models to use in the test extractions

In [8]:
def _first_existing_path(candidates: list[Path]) -> Path:
    for p in candidates:
        if p.exists():
            return p
    return candidates[0]


# Canonical registry location (shared across all notebooks and scripts)
REGISTRY_PATH = Path("../../outputs/model_registry.json")
if not REGISTRY_PATH.exists():
    raise FileNotFoundError(f"Registry not found: {REGISTRY_PATH.resolve()}")


def _norm_rel(path_like: str) -> str:
    return normpath(path_like.replace("\\", "/")).lstrip("./")


def _parse_created_at(value: str) -> datetime:
    if not value:
        return datetime.min.replace(tzinfo=timezone.utc)
    try:
        return datetime.fromisoformat(value.replace("Z", "+00:00"))
    except ValueError:
        return datetime.min.replace(tzinfo=timezone.utc)


def _fix_checkpoint_path(path: str) -> str:
    """Keep canonical checkpoint path from the registry.
    
    aml_submit.sh resolves this path relative to AML_DATASTORE_BASE.
    Stripping Daily_rainfall_sample/ can break checkpoint mounts when
    the datastore base is rooted above that folder.
    """
    return _norm_rel(path)


registry = json.loads(REGISTRY_PATH.read_text(encoding="utf-8"))
# Support both registry schemas (though now we only have one):
# 1) {"checkpoints": [...]} with keys model_slug/training_dataset_path
# 2) {"models": [...]} with keys base_model/dataset
entries = registry.get("checkpoints")
if not isinstance(entries, list):
    entries = registry.get("models", [])

MODEL_JOBS = []
missing = []
expected_dataset_norm = _norm_rel(TRAINING_IMAGES_PATH)

def _model_slug_from_entry(entry: dict) -> str:
    direct = str(entry.get("model_slug", "")).strip()
    if direct:
        return direct
    base_model = str(entry.get("base_model", "")).strip()
    marker = "/outputs/checkpoints/"
    if marker in base_model:
        tail = base_model.split(marker, 1)[1]
        run_slug = tail.split("/", 1)[0]
        parts = run_slug.split("-")
        if (
            len(parts) >= 3
            and len(parts[-2]) == 8
            and len(parts[-1]) == 6
            and parts[-2].isdigit()
            and parts[-1].isdigit()
        ):
            return "-".join(parts[:-2])
        if "-" in run_slug:
            return run_slug.rsplit("-", 1)[0]
    return base_model


for label, model_slug, batch_size, total_shards in MODEL_SETTINGS:
    candidates = []
    for e in entries:
        entry_model = _model_slug_from_entry(e)
        entry_dataset = str(
            e.get("training_dataset_path", e.get("dataset", ""))
        )
        entry_training_mode = str(e.get("training_mode", "")).strip()
        checkpoint_path = e.get("checkpoint_path")

        if (
            entry_model == model_slug
            and _norm_rel(entry_dataset) == expected_dataset_norm
            and (not entry_training_mode or entry_training_mode == "consensus-masked")
            and checkpoint_path
        ):
            candidates.append(e)

    if not candidates:
        missing.append(label)
        continue

    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )

    raw_checkpoint_path = str(best["checkpoint_path"])
    checkpoint_path = _fix_checkpoint_path(raw_checkpoint_path)
    MODEL_JOBS.append((label, checkpoint_path, batch_size, total_shards))
    print(f"{label}: {checkpoint_path}")

if missing:
    raise RuntimeError(
        "Missing fine-tuned checkpoints for: "
        + ", ".join(missing)
        + ". Check DATASET_DIR and ensure model_registry has consensus-masked entries for this dataset."
    )

print(f"\nUsing canonical registry: {REGISTRY_PATH.resolve()}")
print(f"Discovered {len(MODEL_JOBS)} checkpoints")

SmolVLM: Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260629-150922/HuggingFaceTB--SmolVLM2-2.2B-Instruct
Granite4: Daily_rainfall_sample/outputs/checkpoints/ibm-granite--granite-vision-4.1-4b-20260629-150950/ibm-granite--granite-vision-4.1-4b
Gemma3: Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260629-151007/google--gemma-3-4b-it
Gemma4: Daily_rainfall_sample/outputs/checkpoints/google--gemma-4-E4B-it-20260629-151026/google--gemma-4-E4B-it
Ministral: Daily_rainfall_sample/outputs/checkpoints/mistralai--Mistral-Small-3.1-24B-Instruct-2503-20260629-151046/mistralai--Mistral-Small-3.1-24B-Instruct-2503

Using canonical registry: /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/model_registry.json
Discovered 5 checkpoints


In [9]:

# DEBUG: Inspect registry entries for discovered checkpoints
print("=" * 100)
print("DISCOVERED CHECKPOINTS - Registry Details")
print("=" * 100)
for label, checkpoint_path, batch_size, total_shards in MODEL_JOBS:
    # Find the corresponding registry entry
    for e in entries:
        if str(e.get("checkpoint_path", "")) == checkpoint_path:
            print(f"\n{label}:")
            print(f"  Checkpoint path: {checkpoint_path}")
            print(f"  Base model: {e.get('base_model', 'N/A')}")
            print(f"  Training mode: {e.get('training_mode', 'N/A')}")
            print(f"  Dataset: {e.get('dataset', 'N/A')}")
            break
print("=" * 100)
print("\nKey issue: The 'Base model' field contains a local datastore path,")
print("not a Hugging Face Hub ID. The extraction script needs to resolve this.")
print("=" * 100)


DISCOVERED CHECKPOINTS - Registry Details

SmolVLM:
  Checkpoint path: Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-Instruct-20260629-150922/HuggingFaceTB--SmolVLM2-2.2B-Instruct
  Base model: Daily_rainfall_sample/outputs/checkpoints/smolvlm2-20260625-094936/HuggingFaceTB--SmolVLM2-2.2B-Instruct
  Training mode: consensus-masked
  Dataset: consensus_data/1st_order/images

Granite4:
  Checkpoint path: Daily_rainfall_sample/outputs/checkpoints/ibm-granite--granite-vision-4.1-4b-20260629-150950/ibm-granite--granite-vision-4.1-4b
  Base model: Daily_rainfall_sample/outputs/checkpoints/granite4-20260625-095008/ibm-granite--granite-vision-4.1-4b
  Training mode: consensus-masked
  Dataset: consensus_data/1st_order/images

Gemma3:
  Checkpoint path: Daily_rainfall_sample/outputs/checkpoints/google--gemma-3-4b-it-20260629-151007/google--gemma-3-4b-it
  Base model: Daily_rainfall_sample/outputs/checkpoints/gemma3-20260625-095027/google--gemma-3-4b-it
  Training mode: 

## Run extraction jobs with fine-tuned checkpoints

This submits one extraction job per fine-tuned model checkpoint on real test images.

In [10]:
for model_name, checkpoint, batch_size, total_shards in MODEL_JOBS:
    print(f"Submitting {model_name}...")
    subprocess.run(
        [
            "bash",
            "../../scripts/aml_submit.sh",
            "--checkpoint",
            checkpoint,
            "--images-path",
            "test_data/from_Ciara/images",
            "--transcriptions-path",
            "test_data/from_Ciara/transcriptions",
            "--total-shards",
            str(total_shards),
            "--node-gpu-workers",
            str(NODE_GPU_WORKERS),
            "--batch-size",
            str(batch_size),
            "--extraction-registry",
            "../../outputs/extraction_registry.json",
            "extract",
        ],
        check=True,
    )

Submitting SmolVLM...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/from_Ciara/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkpoint: azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs/checkpoints/HuggingFaceTB--SmolVLM2-2.2B-I

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

loyal_longan_9z1tv6hzpr
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260629-155739
  Model: smolvlm
  Dataset: test_data/from_Ciara/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Granite4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/from_Ciara/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

plum_key_9683drmkl4
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260629-155808
  Model: smolvlm
  Dataset: test_data/from_Ciara/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma3...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/from_Ciara/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Checkp

...

coral_garden_5sc9vrmqxd
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260629-155831
  Model: smolvlm
  Dataset: test_data/from_Ciara/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Gemma4...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/from_Ciara/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Ch

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

epic_nail_fwds4xxkqz
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260629-155848
  Model: smolvlm
  Dataset: test_data/from_Ciara/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json
Submitting Ministral...
Workspace:  mlw-llmdatarescue-uksouth-01  (rg-climate-llmdatarescue / 79c7890c-2a30-44ef-aa8d-419d25b7bb8e)
Compute:    A100x8
GPU workers per node: 8
Finetune GPU processes: 8
Grad accum steps (base): 8
Auto-scale grad accum:   true
Model:      smolvlm
Images:     azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/test_data/from_Ciara/images
Outputs:    azureml://subscriptions/79c7890c-2a30-44ef-aa8d-419d25b7bb8e/resourcegroups/rg-climate-llmdatarescue/workspaces/mlw-llmdatarescue-uksouth-01/datastores/large_datastore/paths/Daily_rainfall_sample/outputs

Submitting 1 extract shard(s)...
  Ch

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: Th

quiet_garage_z0j4wt5991
Added extraction registry: ../../outputs/extraction_registry.json
  Run: 20260629-155906
  Model: smolvlm
  Dataset: test_data/from_Ciara/images
Submitted 1 extract job(s).
Extraction registry: ../../outputs/extraction_registry.json


When the jobs have been submitted, the extraction registry will contain run names needed for download and analysis.

Run the next cell to discover run names from the registry, then paste the printed `RUN_NAMES = [...]` block into the following cell.

In [11]:
# Discover run names from extraction registry after submissions complete.
# This cell does NOT write external files. It prints a block to paste into the next cell.

EXTRACTION_REGISTRY_PATH = Path("../../outputs/extraction_registry.json")
TARGET_IMAGES_PATH = "test_data/from_Ciara/images"

registry = json.loads(EXTRACTION_REGISTRY_PATH.read_text(encoding="utf-8"))
entries = registry.get("extractions", [])

RUN_NAMES = []
missing = []
target_images_norm = _norm_rel(TARGET_IMAGES_PATH)

for model_name, checkpoint, _batch_size, _total_shards in MODEL_JOBS:
    candidates = [
        e
        for e in entries
        if _norm_rel(str(e.get("images_path", ""))) == target_images_norm
        and str(e.get("checkpoint_path", "")) == checkpoint
        and e.get("run_name")
    ]
    if not candidates:
        missing.append(model_name)
        continue
    best = max(
        candidates,
        key=lambda e: _parse_created_at(str(e.get("created_at", ""))),
    )
    run_name = str(best["run_name"])
    RUN_NAMES.append(run_name)
    print(f"{model_name}: {run_name}")

if missing:
    raise RuntimeError(
        "Missing extraction runs in registry for: "
        + ", ".join(missing)
        + ". Run extraction submission first, then re-run this cell."
    )

EXTRACTION_DIRS = [f"../../outputs/extractions/{r}" for r in RUN_NAMES]

print("\nCopy this block into the next cell:\n")
print("RUN_NAMES = [")
for run_name in RUN_NAMES:
    print(f'    "{run_name}",')
print("]\n")


SmolVLM: 20260629-155739
Granite4: 20260629-155808
Gemma3: 20260629-155831
Gemma4: 20260629-155848
Ministral: 20260629-155906

Copy this block into the next cell:

RUN_NAMES = [
    "20260629-155739",
    "20260629-155808",
    "20260629-155831",
    "20260629-155848",
    "20260629-155906",
]



In [12]:
# Persistent run names for this notebook.
# Paste the RUN_NAMES block printed by the previous cell here.
RUN_NAMES = [
    "20260629-155739",
    "20260629-155808",
    "20260629-155831",
    "20260629-155848",
    "20260629-155906",
]

EXTRACTION_DIRS = [f"../../outputs/extractions/{r}" for r in RUN_NAMES]

if not RUN_NAMES:
    raise RuntimeError("RUN_NAMES is empty. Run the previous cell, then paste its output here.")

print("Using run names:")
for run_name in RUN_NAMES:
    print("  ", run_name)

Using run names:
   20260629-155739
   20260629-155808
   20260629-155831
   20260629-155848
   20260629-155906


When the jobs have completed successfully, download the extractions so we can analyze them locally.

In [13]:
for run_name in RUN_NAMES:
    subprocess.run(
        ["bash", "../../scripts/aml_download.sh", "--run-name", run_name],
        check=True,
    )

Resolving datastore 'large_datastore' in workspace 'mlw-llmdatarescue-uksouth-01'...


Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260629-155739
         workers: 16


Finished[#############################################################]  100.0000%truct/20260629-155739/DRain_1901-1910_Ciara-5831.json"[]  100.0000%


[
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3613.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3614.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3615.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3616.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3617.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3618.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM2-2.2B-Instruct/20260629-155739/DRain_1901-1910_Ciara-3619.json",
  "Daily_rainfall_sample/outputs/extractions/HuggingFaceTB--SmolVLM

...

[
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3613.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3614.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3615.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3616.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3617.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3618.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3619.json",
  "Daily_rainfall_sample/outputs/extractions/google--gemma-4-E4B-it/20260629-155848/DRain_1901-1910_Ciara-3620.json",
  "Daily_rainfall_sample/outputs/extractions/google--g

Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


Storage account: sallmdatarescue02  container: default

             to:  /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260629-155906
         workers: 16


Finished[#############################################################]  100.0000%-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-5831.json"[]  100.0000%


[
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3613.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3614.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3615.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3616.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3617.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3618.json",
  "Daily_rainfall_sample/outputs/extractions/mistralai--Mistral-Small-3.1-24B-Instruct-2503/20260629-155906/DRain_1901-1910_Ciara-3619.json",
  "D

Build 2nd-order consensus from downloaded extractions and compare to ground truth.

This produces outputs in `outputs/consensus_Ciara_data/test_2nd_order`.

In [14]:
cmd = [
    "bash",
    "../../scripts/run_consensus_pipeline.sh",
    "--dataset-root",
    "../../outputs/consensus_Ciara_data",
    "--images-dir",
    "../../test_data/from_Ciara/images",
    "--variant-name",
    "test_2nd_order",
    "--threshold",
    "3",
    "--null-threshold",
    "5",
    "--precision",
    "3",
]

for extraction_dir in EXTRACTION_DIRS:
    cmd.extend(["--extraction-dir", extraction_dir])

cmd.extend(
    [
        "--validate",
        "--ground-truth-dir",
        "../../test_data/from_Ciara/transcriptions",
        "--validation-sample-denominator",
        "1",
    ]
)

subprocess.run(cmd, check=True)

=== Consensus Pipeline ===
Dataset root:      ../../outputs/consensus_Ciara_data
Variant name:      test_2nd_order
Threshold:         3
Null threshold:    5
Precision:         3
Extraction dirs:   5
Validate:          true
Validation sample: 1/1

Creating config...
✓ Created consensus config
  Variant:           test_2nd_order
  Dataset root:      /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_Ciara_data
  Variant dir:       /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_Ciara_data/test_2nd_order
  Config file:       /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_Ciara_data/test_2nd_order/consensus_config.json
  Extraction dirs:   5 model(s)
                     1. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260629-155739
                     2. /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/extractions/20260629-155808
                     3. /ho

CompletedProcess(args=['bash', '../../scripts/run_consensus_pipeline.sh', '--dataset-root', '../../outputs/consensus_Ciara_data', '--images-dir', '../../test_data/from_Ciara/images', '--variant-name', 'test_2nd_order', '--threshold', '3', '--null-threshold', '5', '--precision', '3', '--extraction-dir', '../../outputs/extractions/20260629-155739', '--extraction-dir', '../../outputs/extractions/20260629-155808', '--extraction-dir', '../../outputs/extractions/20260629-155831', '--extraction-dir', '../../outputs/extractions/20260629-155848', '--extraction-dir', '../../outputs/extractions/20260629-155906', '--validate', '--ground-truth-dir', '../../test_data/from_Ciara/transcriptions', '--validation-sample-denominator', '1'], returncode=0)

## Validate each individual extraction against ground truth

This computes per-model extraction quality directly against Ciara's ground-truth transcriptions.

In [15]:
labels = [name for name, _checkpoint, _batch, _shards in MODEL_JOBS]

cmd = [
    "python",
    "../../scripts/evaluate_extraction_quality.py",
    "--ground-truth-dir",
    "../../test_data/from_Ciara/transcriptions",
    "--output-json",
    "../../outputs/consensus_Ciara_data/test_2nd_order/extraction_quality_summary.json",
]

for label, extraction_dir in zip(labels, EXTRACTION_DIRS):
    cmd.extend(["--extraction-dir", extraction_dir, "--label", label])

subprocess.run(cmd, check=True)

print("\nSaved quality summary to:")
print(Path("../../outputs/consensus_Ciara_data/test_2nd_order/extraction_quality_summary.json").resolve())


EXTRACTION QUALITY SUMMARY
Label         Acc(all) Acc(eval)  Coverage  Correct  Incorrect Cells(eval)   Miss    Bad
----------------------------------------------------------------------------------------
Granite4        0.9510    0.9510    1.0000    23373       1203       24576      0      0
Ministral       0.9410    0.9410    1.0000    23125       1451       24576      0      0
Gemma3          0.8997    0.8997    1.0000    22111       2465       24576      0      0
Gemma4          0.8914    0.8914    1.0000    21907       2669       24576      0      0
SmolVLM         0.8613    0.8750    0.9844    21167       3025       24192      0      1

Wrote summary JSON: /home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_Ciara_data/test_2nd_order/extraction_quality_summary.json

Saved quality summary to:
/home/users/philip.brohan/Projects/Auto-Daily-Rainfall-MO/outputs/consensus_Ciara_data/test_2nd_order/extraction_quality_summary.json


In [16]:
import json
from pathlib import Path


def _load_summary(path_str: str) -> list[dict]:
    path = Path(path_str)
    if not path.exists():
        raise FileNotFoundError(f"Summary file not found: {path}")
    with path.open("r", encoding="utf-8") as f:
        data = json.load(f)
    if isinstance(data, dict):
        if "results" in data and isinstance(data["results"], list):
            return data["results"]
        if "rows" in data and isinstance(data["rows"], list):
            return data["rows"]
    if isinstance(data, list):
        return data
    raise ValueError(f"Unexpected summary format in {path}")


def _index_by_label(rows: list[dict]) -> dict[str, dict]:
    out = {}
    for r in rows:
        label = str(r.get("label", "")).strip()
        if label:
            out[label] = r
    return out


summary_1st = _load_summary("../../outputs/consensus_Ciara_data/test_1st_order/extraction_quality_summary.json")
summary_2nd = _load_summary("../../outputs/consensus_Ciara_data/test_2nd_order/extraction_quality_summary.json")

by_1 = _index_by_label(summary_1st)
by_2 = _index_by_label(summary_2nd)
labels = sorted(set(by_1.keys()) | set(by_2.keys()))

metrics = [
    "accuracy_vs_all_ground_truth_cells",
    "accuracy_on_evaluated_cells",
    "coverage_of_ground_truth_cells",
]

print("1ST vs 2ND ORDER EXTRACTION QUALITY")
print("=" * 120)
header = (
    f"{'Model':<12} "
    f"{'Acc(all) 1st':>12} {'Acc(all) 2nd':>12} {'Delta':>10} "
    f"{'Acc(eval) 1st':>14} {'Acc(eval) 2nd':>14} {'Delta':>10} "
    f"{'Coverage 1st':>12} {'Coverage 2nd':>12} {'Delta':>10}"
)
print(header)
print("-" * len(header))

for label in labels:
    r1 = by_1.get(label, {})
    r2 = by_2.get(label, {})

    def _val(row: dict, key: str):
        v = row.get(key)
        return float(v) if isinstance(v, (int, float)) else None

    a1 = _val(r1, metrics[0])
    e1 = _val(r1, metrics[1])
    c1 = _val(r1, metrics[2])
    a2 = _val(r2, metrics[0])
    e2 = _val(r2, metrics[1])
    c2 = _val(r2, metrics[2])

    def _fmt(v):
        return f"{v:0.4f}" if isinstance(v, float) else "N/A"

    def _d(v1, v0):
        if isinstance(v1, float) and isinstance(v0, float):
            return v1 - v0
        return None

    print(
        f"{label:<12} "
        f"{_fmt(a1):>12} {_fmt(a2):>12} {_fmt(_d(a2, a1)):>10} "
        f"{_fmt(e1):>14} {_fmt(e2):>14} {_fmt(_d(e2, e1)):>10} "
        f"{_fmt(c1):>12} {_fmt(c2):>12} {_fmt(_d(c2, c1)):>10}"
    )

print("\nLegend: Delta = 2nd_order - 1st_order")
print("Positive deltas indicate improvement in 2nd-order over 1st-order.")

1ST vs 2ND ORDER EXTRACTION QUALITY
Model        Acc(all) 1st Acc(all) 2nd      Delta  Acc(eval) 1st  Acc(eval) 2nd      Delta Coverage 1st Coverage 2nd      Delta
-------------------------------------------------------------------------------------------------------------------------------
Gemma3             0.6893       0.8997     0.2104         0.7003         0.8997     0.1994       0.9844       1.0000     0.0156
Gemma4             0.5349       0.8914     0.3565         0.6583         0.8914     0.2331       0.8125       1.0000     0.1875
Granite4           0.9082       0.9510     0.0428         0.9226         0.9510     0.0284       0.9844       1.0000     0.0156
Ministral          0.8383       0.9410     0.1026         0.8516         0.9410     0.0893       0.9844       1.0000     0.0156
SmolVLM            0.5384       0.8613     0.3229         0.6756         0.8750     0.1994       0.7969       0.9844     0.1875

Legend: Delta = 2nd_order - 1st_order
Positive deltas indicate impr